# 06 The Fundamental Product of Geometric Algebra

This standalone notebook treats Chapter 6 as one executable lab. The through-line is simple: the geometric product keeps the metric part and the oriented-spanning part of a vector product in one invertible expression. Once those pieces live together, division, projection, reflection, and operator ratios become ordinary algebraic moves instead of separate geometric recipes.

The source heading audit for printed pages 141-166 is saved in `D:/Geometry/artifacts/chapter-06/headings.json`. In this PDF scan, Chapter 6 has extractable text on printed pages 141-165; the next extractable text page is Chapter 7 on printed page 167, so printed page 166 appears blank in the scan.

The chapter is also a change in programming style. Before this point it is tempting to write geometry code as a menu of unrelated formulas: one formula for area, another for projection, another for reflection, another for frame construction. The geometric product encourages a smaller set of primitives. Multiply, select grades, and divide by invertible elements. That does not make the formulas disappear, but it gives them a shared implementation path and a shared set of tests.

A useful way to read every cell is to ask what information is being preserved. If we keep only a scalar, orientation is gone. If we keep only a blade of span, metric alignment is gone. The geometric product keeps both when both are relevant. That is why the later cells can solve equations, build operators, and split vectors without switching mathematical languages.


## Lab Map

Work through the cells in order. Each section introduces a calculation, then asks the code to check an invariant. The interactive cells are not decoration: move the controls until a sign or grade surprises you, then inspect the algebraic output immediately below it.

Covered themes:

- vector geometric product
- symmetry and antisymmetry
- basis-vector rules
- vector division
- vector ratios as operators
- multivector geometric product
- retrieving subspace products by grade selection
- projection, rejection, and reflection
- Gram-Schmidt orthogonalization as a blade-division lab

The saved artifacts are part of the lesson rather than side effects. Static figures are written for quick review outside the notebook. JSON reports are written so a later audit can check the basis table, grade split, headings, and quality gate without re-reading the prose. The interactive widgets are included for exploration, while the saved figures give stable snapshots for validation.

The examples stay in Euclidean 3-D because the signs and grades are already rich enough there. Non-Euclidean metrics introduce null vectors and null blades, which are important but would distract from the first purpose of the chapter: understand why the single product is invertible for ordinary non-null elements and why that invertibility matters.


In [ ]:
from pathlib import Path
import json
import math
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

PROJECT_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "utils" / "chapter06_geometric_product.py").exists():
        PROJECT_ROOT = candidate
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.artifacts import save_json, save_matplotlib, save_plotly_html
from utils.chapter06_geometric_product import (
    E3,
    apply_right_ratio,
    basis_product_table,
    coords_to_vector,
    gram_schmidt_ga,
    pairwise_inner_matrix,
    project_onto_blade,
    pure_grade,
    reflect_in_blade,
    reject_from_blade,
    right_ratio,
    vector_inverse,
    vector_product_parts,
    vector_to_coords,
)

np.set_printoptions(precision=4, suppress=True)
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
ARTIFACT_TOPIC = "chapter-06"
CHAPTER_DIR = PROJECT_ROOT / "Geometric-Algebra-for-Computer-Science" / "part-01-geometric-algebra" / "chapter-06-the-fundamental-product-of-geometric-algebra"
NOTEBOOK_PATH = CHAPTER_DIR / "06-the-fundamental-product-of-geometric-algebra.ipynb"
e1, e2, e3 = E3.basis()
zero = E3.scalar(0)


In [ ]:
headings_path = PROJECT_ROOT / "artifacts" / "chapter-06" / "headings.json"
heading_audit = json.loads(headings_path.read_text(encoding="utf-8"))
pd.DataFrame({"verified_heading": heading_audit["headings"]})


## Vector Product: Two Kinds of Information

For vectors, the geometric product `a.gp(b)` is deliberately mixed-grade. Its scalar part measures alignment. Its bivector part records oriented area in the plane swept from `a` toward `b`. Keeping both terms is the whole trick: neither the inner product nor the outer product alone can be inverted, but their sum remembers enough to divide by a non-null vector.

The symmetric half of the product is the metric part. The antisymmetric half is the oriented-area part. The code below checks that the two retrieval methods agree.

Notice that the product is not being defined as a trick for producing two familiar answers at once. It is better to think of it as the natural object that those two familiar answers were separately sampling. The scalar part and bivector part are projections of one richer product into different grades. That is why the notebook first computes `a.gp(b)` and only then asks for pieces. The full product is the source of truth.

This distinction matters in later operator formulas. If you prematurely discard the bivector part, a vector ratio loses its turning information. If you prematurely discard the scalar part, division loses the alignment information needed to recover a vector. The product works because the pieces remain available until a later computation has enough context to choose what it needs.


In [ ]:
a = coords_to_vector([1.5, 0.5, 0.0])
b = coords_to_vector([0.25, 1.25, 0.0])
parts = vector_product_parts(a, b)

assert parts["geometric"].almost_equal(parts["inner"] + parts["outer"])
assert parts["symmetric"].almost_equal(parts["inner"])
assert parts["antisymmetric"].almost_equal(parts["outer"])
assert a.gp(b).almost_equal(a.scalar_product(b) + a.wedge(b))

pd.DataFrame(
    [
        {"piece": name, "value": repr(value), "grades": sorted(value.grades())}
        for name, value in parts.items()
    ]
)


In [ ]:
def vector_product_figure(theta_degrees=55, length_b=1.25):
    angle = math.radians(theta_degrees)
    a_xy = np.array([1.0, 0.0])
    b_xy = length_b * np.array([math.cos(angle), math.sin(angle)])
    A = coords_to_vector([a_xy[0], a_xy[1], 0.0])
    B = coords_to_vector([b_xy[0], b_xy[1], 0.0])
    split = vector_product_parts(A, B)
    inner = split["inner"].scalar_value()
    outer = split["outer"].terms.get(0b011, 0.0)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.axhline(0, color="0.88", linewidth=1)
    ax.axvline(0, color="0.88", linewidth=1)
    ax.quiver(0, 0, *a_xy, angles="xy", scale_units="xy", scale=1, color="#1f77b4", label="a")
    ax.quiver(0, 0, *b_xy, angles="xy", scale_units="xy", scale=1, color="#d62728", label="b")
    parallelogram = np.array([[0, 0], a_xy, a_xy + b_xy, b_xy, [0, 0]])
    ax.fill(parallelogram[:, 0], parallelogram[:, 1], color="#2ca02c", alpha=0.18)
    ax.plot(parallelogram[:, 0], parallelogram[:, 1], color="#2ca02c", linewidth=1)
    ax.text(0.02, -0.28, f"scalar part: {inner:.3f}\nbivector e1^e2 part: {outer:.3f}", fontsize=10)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-1.4, 1.8)
    ax.set_ylim(-1.4, 1.8)
    ax.legend(loc="upper right")
    ax.set_title("Vector product: alignment plus oriented area")
    return fig

fig = vector_product_figure()
vector_product_png = save_matplotlib(fig, ARTIFACT_TOPIC, "visuals", "vector-product.png", root=ARTIFACT_ROOT)
display(fig)
plt.close(fig)
print(f"wrote {vector_product_png}")

def vector_product_scene(theta_degrees=55, length_b=1.25):
    fig = vector_product_figure(theta_degrees, length_b)
    display(fig)
    plt.close(fig)

widgets.interact(
    vector_product_scene,
    theta_degrees=widgets.IntSlider(value=55, min=-170, max=170, step=5, description="angle"),
    length_b=widgets.FloatSlider(value=1.25, min=0.2, max=2.0, step=0.05, description="|b|"),
)


## Basis Rules

In an orthonormal Euclidean basis, equal basis vectors square to `1`, while different basis vectors anticommute. That tiny rule set is enough to compute larger products by expansion and associativity. The bivector `e1.gp(e2)` behaves like an oriented plane element: under the geometric product its square is `-1`, not because it is a complex number, but because a quarter-turn plane element applied twice reverses direction.

The table cell is deliberately small, but it is the engine room. Every larger product in this notebook is a distributed sum of these basis products. When two basis blades share a basis vector, that vector pair collapses through the metric. When they do not share a vector, the result grows by span and receives the sign required to return to canonical basis order. The implementation in `utils.ga` stores basis blades as bit masks, so the same rule becomes a deterministic bit operation.

A common beginner mistake is to treat the minus signs as algebraic nuisance. In geometric algebra, the signs are orientation data. Reversing a bivector is not a formatting change; it reverses the oriented plane element. That is why the basis table belongs in a chapter about geometry, not only in a chapter about symbolic manipulation.


In [ ]:
table_rows = basis_product_table(E3)
table = pd.DataFrame(table_rows).pivot(index="left", columns="right", values="product")
table_json = save_json(table_rows, ARTIFACT_TOPIC, "tables", "basis-product-table.json", root=ARTIFACT_ROOT)
display(table)

e12 = e1.gp(e2)
assert e1.gp(e1).almost_equal(E3.scalar(1))
assert e2.gp(e1).almost_equal(-e1.gp(e2))
assert e12.gp(e12).almost_equal(E3.scalar(-1))
print(f"wrote {table_json}")
{"e12": repr(e12), "e12_squared": repr(e12.gp(e12)), "grade": pure_grade(e12)}


## Vector Division

A non-null vector has a geometric-product inverse: multiply it by itself and only a scalar remains. Right division by a vector means multiplying on the right by that inverse. Because the product is not generally commutative, the side matters. The next check multiplies an unknown vector by `a`, then divides on the same side to recover it.

Division also gives a good test of whether the product has been implemented honestly. If the basis signs are wrong, if the metric squares are wrong, or if grade filtering accidentally drops a term before division, the recovered vector will not match the original. The check in this section is intentionally plain: encode a vector by multiplying on the right, decode it by multiplying on the right by the inverse, and compare the result. A healthy implementation should make that feel as routine as multiplying and dividing numbers.

The inverse used here is a vector inverse, not a matrix inverse. Nothing global about the coordinate system has to be solved. The vector itself supplies the inverse because its square is scalar. That local algebraic inverse is one of the reasons geometric algebra can make geometric equations look like ordinary arithmetic while still preserving grade structure.


In [ ]:
x = coords_to_vector([1.0, -0.5, 0.25])
divisor = coords_to_vector([2.0, 1.0, 0.0])
encoded = x.gp(divisor)
recovered = encoded.gp(vector_inverse(divisor)).grade(1)

assert recovered.almost_equal(x)
assert vector_inverse(divisor).gp(divisor).almost_equal(E3.scalar(1))
assert divisor.gp(vector_inverse(divisor)).almost_equal(E3.scalar(1))

pd.DataFrame(
    [
        {"quantity": "x", "value": repr(x)},
        {"quantity": "divisor", "value": repr(divisor)},
        {"quantity": "x divisor", "value": repr(encoded)},
        {"quantity": "(x divisor) divisor^{-1}", "value": repr(recovered)},
    ]
)


## Ratios of Vectors as Operators

A vector ratio can be read operationally: build the factor that carries one vector to another, then let nearby vectors ride along. In this notebook the ratio acts on the right, so the operator that sends `source` to `target` is `source^{-1} target`, and a test vector moves as `x source^{-1} target`.

This is a compact way to mix scale and rotation in a plane. It is also a warning label: a product can be a geometric object and an action at the same time.

The ratio picture is also a preview of versors. A single vector ratio is not yet the full sandwich machinery of the next chapter, but it already shows a central theme: products of invertible vectors can act on geometry. The operator is not stored as a matrix. It is stored as an algebra element whose multiplication behavior performs the transformation.

This viewpoint helps explain why the chapter places ratios before the full treatment of reflections and rotations. Ratios are a bridge from solving equations to thinking operationally. Once you believe that one product can carry both measurement and turning information, it is natural to let that product act on more than the vector used to define it. The interactive grid makes the leap concrete: every point of the grid receives the same ratio, so the local vector equation becomes a global plane action.


In [ ]:
source = e1
target = 1.8 * e2
probe = e1 + e2
ratio = right_ratio(source, target)
moved_source = apply_right_ratio(source, source, target)
moved_probe = apply_right_ratio(probe, source, target)

assert moved_source.almost_equal(target)
assert abs(moved_probe.norm() - probe.norm() * target.norm() / source.norm()) < 1e-9

pd.DataFrame(
    [
        {"quantity": "source", "value": repr(source)},
        {"quantity": "target", "value": repr(target)},
        {"quantity": "source^{-1} target", "value": repr(ratio)},
        {"quantity": "probe", "value": repr(probe)},
        {"quantity": "probe after ratio", "value": repr(moved_probe)},
    ]
)


In [ ]:
def make_ratio_figure(angle_degrees=80, scale=1.4):
    angle = math.radians(angle_degrees)
    target = coords_to_vector([scale * math.cos(angle), scale * math.sin(angle), 0.0])

    fig = go.Figure()
    grid_values = np.linspace(-1.0, 1.0, 5)
    samples = np.linspace(-1.0, 1.0, 41)

    def transform_point(point):
        vector = coords_to_vector([point[0], point[1], 0.0])
        return vector_to_coords(apply_right_ratio(vector, e1, target))[:2]

    for value in grid_values:
        horizontal = np.array([[t, value] for t in samples])
        vertical = np.array([[value, t] for t in samples])
        horizontal_t = np.array([transform_point(point) for point in horizontal])
        vertical_t = np.array([transform_point(point) for point in vertical])
        fig.add_trace(go.Scatter(x=horizontal[:, 0], y=horizontal[:, 1], mode="lines", line=dict(color="#b8c0cc", width=1), showlegend=False))
        fig.add_trace(go.Scatter(x=vertical[:, 0], y=vertical[:, 1], mode="lines", line=dict(color="#b8c0cc", width=1), showlegend=False))
        fig.add_trace(go.Scatter(x=horizontal_t[:, 0], y=horizontal_t[:, 1], mode="lines", line=dict(color="#2ca02c", width=2), showlegend=False))
        fig.add_trace(go.Scatter(x=vertical_t[:, 0], y=vertical_t[:, 1], mode="lines", line=dict(color="#2ca02c", width=2), showlegend=False))

    target_xy = vector_to_coords(target)[:2]
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 0], mode="lines+markers", name="source", line=dict(color="#1f77b4", width=4)))
    fig.add_trace(go.Scatter(x=[0, target_xy[0]], y=[0, target_xy[1]], mode="lines+markers", name="target", line=dict(color="#d62728", width=4)))
    fig.update_layout(
        title="Right vector ratio: source^{-1} target",
        width=650,
        height=540,
        xaxis=dict(scaleanchor="y", scaleratio=1, range=[-2.6, 2.6], zeroline=True),
        yaxis=dict(range=[-2.6, 2.6], zeroline=True),
        margin=dict(l=30, r=20, t=55, b=30),
    )
    return fig

ratio_figure = make_ratio_figure()
ratio_html = save_plotly_html(
    ratio_figure,
    ARTIFACT_TOPIC,
    "visuals",
    "ratio-operator.html",
    root=ARTIFACT_ROOT,
    include_plotlyjs="cdn",
)
display(ratio_figure)
print(f"wrote {ratio_html}")


In [ ]:
def ratio_widget(angle_degrees=80, scale=1.4):
    display(make_ratio_figure(angle_degrees, scale))

widgets.interact(
    ratio_widget,
    angle_degrees=widgets.IntSlider(value=80, min=-170, max=170, step=5, description="angle"),
    scale=widgets.FloatSlider(value=1.4, min=0.25, max=2.5, step=0.05, description="scale"),
)


## Multivector Geometric Product

For general multivectors, the geometric product is extended by linearity, distributivity, and associativity. That means a product may contain several grades at once. This is not noise. The grade pattern tells us which subspace relationships were present in the operands.

A practical habit is to compute the product once, then inspect selected grades. That gives a uniform way to recover familiar products without giving them separate implementations at every call site.

Mixed-grade products can feel untidy until you stop expecting a product to have one grade. A multivector is a sum of geometric roles. Multiplying two such sums means every role in the left operand interacts with every role in the right operand. Some interactions lower grade because directions cancel through the metric. Some raise grade because independent directions span something larger. Some preserve grade because the two effects balance.

The right question is not, why did this product create several grades? The right question is, which grade answers my geometric question? This is why the notebook reports the grade components of `A B` and `B A`. The total product is useful, but so are its projections onto individual grades. In a larger application, those projections become the familiar subspace operations, operator components, and scalar invariants that tests can inspect.


In [ ]:
A = E3.scalar(1.0) + 2.0 * e1 - 0.5 * e2 + 0.75 * e1.wedge(e2)
B = -0.25 * E3.scalar(1.0) + 0.5 * e1 + 1.25 * e3 - 0.5 * e2.wedge(e3)
C = e1 - e2 + e3

AB = A.gp(B)
BA = B.gp(A)
assert A.gp(B).gp(C).almost_equal(A.gp(B.gp(C)))

def grade_components(multivector):
    return {
        f"grade_{grade}": repr(multivector.grade(grade))
        for grade in range(E3.dimension + 1)
        if multivector.grade(grade).terms
    }

pd.DataFrame(
    [
        {"product": "A B", "value": repr(AB), **grade_components(AB)},
        {"product": "B A", "value": repr(BA), **grade_components(BA)},
    ]
)


## Retrieving Subspace Products

For a vector times a blade, low grades carry contraction-like information and high grades carry spanning information. The example below multiplies a vector by the `e1^e2` plane. The grade-1 part is the component of the vector that interacts metrically with the plane. The grade-3 part is the volume created by the component that sticks out of the plane.

Retrieval by grade selection is one of the strongest arguments for keeping the implementation close to the algebra. If the code already knows how to multiply and select grades, then contraction-like and span-like operations are not separate mysteries. They are recognizable slices of one product. This reduces the number of places where sign conventions can diverge.

The example uses a vector and a plane because the geometry is visible. A vector has an in-plane component and an out-of-plane component. Multiplication by the plane creates two grades, and those grades correspond to the two ways the vector can relate to the plane. The in-plane part participates in a metric relation. The out-of-plane part creates volume. Grade selection reads those facts back out.


In [ ]:
v = coords_to_vector([1.0, 2.0, 3.0])
plane = e1.wedge(e2)
product = v.gp(plane)
contracted = v.left_contract(plane)
spanned = v.wedge(plane)

assert product.grade(1).almost_equal(contracted)
assert product.grade(3).almost_equal(spanned)
assert product.almost_equal(contracted + spanned)

grade_report = [
    {"quantity": "v plane", "value": repr(product), "grades": sorted(product.grades())},
    {"quantity": "selected grade 1", "value": repr(product.grade(1)), "grades": [1]},
    {"quantity": "left contraction", "value": repr(contracted), "grades": sorted(contracted.grades())},
    {"quantity": "selected grade 3", "value": repr(product.grade(3)), "grades": [3]},
    {"quantity": "outer product", "value": repr(spanned), "grades": sorted(spanned.grades())},
]
grade_json = save_json(grade_report, ARTIFACT_TOPIC, "tables", "grade-selection-report.json", root=ARTIFACT_ROOT)
print(f"wrote {grade_json}")
pd.DataFrame(grade_report)


## Projection, Rejection, and Reflection

Division by a blade lets us decompose a vector relative to a subspace. Projection is the part that lies in the blade. Rejection is the part that is perpendicular to it. Reflection keeps the projection and reverses the rejection. The checks below use the `e1^e2` plane, then the interactive cell repeats the idea for a movable line in the plane.

The decomposition also clarifies the relationship between projection and reflection. Projection and rejection are complementary pieces of the same vector. Reflection is not a new decomposition; it is a sign change on one piece. In a coordinate-only treatment those operations may be written with different formulas. In the geometric-product treatment they share the same subspace blade and inverse, which makes their relationship hard to miss.

The formula works for a line, a plane, or a higher-dimensional blade as long as the blade is invertible. That uniformity is the real payoff. The code does not need to know whether the subspace is represented by a single vector or a bivector before deciding the overall strategy. It computes the grade-sensitive products, divides by the blade, and then checks that projection plus rejection returns the original vector.


In [ ]:
x = coords_to_vector([1.0, -0.5, 2.0])
plane = e1.wedge(e2)
projected = project_onto_blade(x, plane)
rejected = reject_from_blade(x, plane)
reflected = reflect_in_blade(x, plane)

assert (projected + rejected).almost_equal(x)
assert reflected.almost_equal(projected - rejected)
assert projected.scalar_product(rejected).almost_equal(zero)

pd.DataFrame(
    [
        {"quantity": "x", "value": repr(x)},
        {"quantity": "projection onto e1^e2", "value": repr(projected)},
        {"quantity": "rejection from e1^e2", "value": repr(rejected)},
        {"quantity": "reflection in e1^e2", "value": repr(reflected)},
    ]
)


In [ ]:
def projection_figure(line_angle_degrees=30, x_x=1.2, x_y=0.7):
    angle = math.radians(line_angle_degrees)
    line = coords_to_vector([math.cos(angle), math.sin(angle), 0.0])
    x = coords_to_vector([x_x, x_y, 0.0])
    p = project_onto_blade(x, line)
    r = reject_from_blade(x, line)
    f = reflect_in_blade(x, line)

    x_xy = vector_to_coords(x)[:2]
    p_xy = vector_to_coords(p)[:2]
    r_xy = vector_to_coords(r)[:2]
    f_xy = vector_to_coords(f)[:2]
    line_xy = vector_to_coords(line)[:2]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axhline(0, color="0.9", linewidth=1)
    ax.axvline(0, color="0.9", linewidth=1)
    ax.plot([-2 * line_xy[0], 2 * line_xy[0]], [-2 * line_xy[1], 2 * line_xy[1]], color="#111111", linewidth=1.5, label="subspace")
    ax.quiver(0, 0, *x_xy, angles="xy", scale_units="xy", scale=1, color="#1f77b4", label="x")
    ax.quiver(0, 0, *p_xy, angles="xy", scale_units="xy", scale=1, color="#2ca02c", label="projection")
    ax.quiver(p_xy[0], p_xy[1], *r_xy, angles="xy", scale_units="xy", scale=1, color="#ff7f0e", label="rejection")
    ax.quiver(0, 0, *f_xy, angles="xy", scale_units="xy", scale=1, color="#d62728", label="reflection")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-2.2, 2.2)
    ax.set_ylim(-2.2, 2.2)
    ax.legend(loc="upper right")
    ax.set_title("Projection plus rejection; reflection flips rejection")
    return fig

fig = projection_figure()
projection_png = save_matplotlib(fig, ARTIFACT_TOPIC, "visuals", "projection-reflection.png", root=ARTIFACT_ROOT)
display(fig)
plt.close(fig)
print(f"wrote {projection_png}")

def projection_scene(line_angle_degrees=30, x_x=1.2, x_y=0.7):
    fig = projection_figure(line_angle_degrees, x_x, x_y)
    display(fig)
    plt.close(fig)

widgets.interact(
    projection_scene,
    line_angle_degrees=widgets.IntSlider(value=30, min=-90, max=90, step=5, description="line"),
    x_x=widgets.FloatSlider(value=1.2, min=-1.8, max=1.8, step=0.1, description="x1"),
    x_y=widgets.FloatSlider(value=0.7, min=-1.8, max=1.8, step=0.1, description="x2"),
)


## Gram-Schmidt as Repeated Rejection

Classical Gram-Schmidt subtracts projections. The geometric-algebra version says the same thing in a blade-native way: maintain the blade already spanned, wedge in the next input vector, then divide by the old blade to straighten the new perpendicular contribution back into a vector. The result is an orthogonal frame whose vectors still span the same growing subspaces when the input vectors are independent.

The Gram-Schmidt lab is included because it is a programming example with teeth. It combines outer products, division by blades, grade selection, and numerical checks in one workflow. It also demonstrates a useful pattern for building algorithms in geometric algebra: keep an accumulating geometric object, update it by a product, and recover the grade needed for the next step.

The first output vector is just the first input vector, viewed as the rejection from the scalar subspace. The second output vector is the part of the second input perpendicular to the first. The third output vector is the part of the third input perpendicular to the plane already spanned by the first two outputs. At each step the current blade records the subspace that has already been accepted. Dividing by that blade turns the newly created higher-grade object back into the next orthogonal vector.

The inner-product matrix is the audit trail. If the off-diagonal entries are near zero, the frame is orthogonal. If a future edit changes the order of the wedge or the side of division, those off-diagonal entries will usually light up immediately. That makes this lab a compact regression test, not just an illustration.


In [ ]:
inputs = [
    coords_to_vector([1.0, 0.4, 0.2]),
    coords_to_vector([0.2, 1.1, 0.3]),
    coords_to_vector([0.3, -0.2, 1.2]),
]
frame = gram_schmidt_ga(inputs)
inner_matrix = pairwise_inner_matrix(frame)

assert np.allclose(inner_matrix - np.diag(np.diag(inner_matrix)), 0.0, atol=1e-9)
assert all(vector.norm2() > 1e-9 for vector in frame)

display(pd.DataFrame({"input": [repr(v) for v in inputs], "orthogonalized": [repr(v) for v in frame]}))
pd.DataFrame(inner_matrix, columns=["b1", "b2", "b3"], index=["b1", "b2", "b3"])


In [ ]:
def gram_schmidt_figure(skew=0.35, lift=0.8):
    vectors = [
        coords_to_vector([1.0, 0.0, 0.0]),
        coords_to_vector([skew, 1.0, 0.0]),
        coords_to_vector([0.25, skew, lift]),
    ]
    frame = gram_schmidt_ga(vectors)
    original = np.array([vector_to_coords(v) for v in vectors])
    orthogonal = np.array([vector_to_coords(v) for v in frame])

    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")
    for vec, label in zip(original, ["v1", "v2", "v3"]):
        ax.quiver(0, 0, 0, vec[0], vec[1], vec[2], color="#9ecae1", linewidth=2, arrow_length_ratio=0.08)
        ax.text(vec[0], vec[1], vec[2], label, color="#5b8bb1")
    for vec, color, label in zip(orthogonal, ["#1f77b4", "#2ca02c", "#d62728"], ["b1", "b2", "b3"]):
        ax.quiver(0, 0, 0, vec[0], vec[1], vec[2], color=color, linewidth=3, arrow_length_ratio=0.08)
        ax.text(vec[0], vec[1], vec[2], label, color=color)
    ax.set_xlim(-0.2, 1.4)
    ax.set_ylim(-0.2, 1.4)
    ax.set_zlim(-0.2, 1.4)
    ax.set_title("Blade-division Gram-Schmidt")
    ax.set_xlabel("e1")
    ax.set_ylabel("e2")
    ax.set_zlabel("e3")
    return fig

fig = gram_schmidt_figure()
gs_png = save_matplotlib(fig, ARTIFACT_TOPIC, "visuals", "gram-schmidt-frame.png", root=ARTIFACT_ROOT)
display(fig)
plt.close(fig)
print(f"wrote {gs_png}")

def gram_schmidt_scene(skew=0.35, lift=0.8):
    fig = gram_schmidt_figure(skew, lift)
    display(fig)
    plt.close(fig)

widgets.interact(
    gram_schmidt_scene,
    skew=widgets.FloatSlider(value=0.35, min=-0.8, max=0.95, step=0.05, description="skew"),
    lift=widgets.FloatSlider(value=0.8, min=0.2, max=1.6, step=0.05, description="lift"),
)


## Sanity Checks

A good implementation should make the chapter claims cheap to test. The final cell collects the invariants used above and writes a small JSON report into the chapter artifact folder. If a future edit breaks a sign convention, a grade selection, or a division side, at least one assertion here should fail loudly.

The quality gate is intentionally placed inside the notebook. That may seem unusual, but it prevents a common failure mode in generated course material: a notebook can execute while still being too shallow to teach. Here the final cell checks both mathematical behavior and instructional substance. It reads the notebook file, counts markdown words and code cells, and counts saved artifacts in the chapter artifact directory.

Those quality checks are not substitutes for human review. They simply establish a floor. The prose still needs to be coherent, the examples still need to be meaningful, and the artifacts still need to correspond to the claims being made. The checks make sure the floor stays in place when later workers revise the chapter.


In [ ]:
sanity_a = coords_to_vector([0.8, -1.2, 0.4])
sanity_b = coords_to_vector([1.1, 0.3, -0.6])
sanity_c = coords_to_vector([-0.2, 0.7, 1.3])
sanity_plane = e1.wedge(e2)
sanity_x = coords_to_vector([0.4, -0.9, 1.1])

split = vector_product_parts(sanity_a, sanity_b)
frame = gram_schmidt_ga([sanity_a, sanity_b, sanity_c])
projection = project_onto_blade(sanity_x, sanity_plane)
rejection = reject_from_blade(sanity_x, sanity_plane)
reflection = reflect_in_blade(sanity_x, sanity_plane)
frame_matrix = pairwise_inner_matrix(frame)

checks = {
    "headings_verified": len(heading_audit["headings"]),
    "vector_gp_splits_into_inner_plus_outer": split["geometric"].almost_equal(split["inner"] + split["outer"]),
    "symmetric_half_is_scalar_part": split["symmetric"].almost_equal(split["inner"]),
    "antisymmetric_half_is_bivector_part": split["antisymmetric"].almost_equal(split["outer"]),
    "basis_bivector_square": repr(e1.gp(e2).gp(e1.gp(e2))),
    "right_division_recovers_vector": sanity_a.gp(sanity_b).gp(vector_inverse(sanity_b)).grade(1).almost_equal(sanity_a),
    "ratio_maps_source_to_target": apply_right_ratio(e1, e1, 1.5 * e2).almost_equal(1.5 * e2),
    "multivector_gp_is_associative_on_sample": sanity_a.gp(sanity_b).gp(sanity_c).almost_equal(sanity_a.gp(sanity_b.gp(sanity_c))),
    "vector_blade_grade_selection": sanity_x.gp(sanity_plane).grade(1).almost_equal(sanity_x.left_contract(sanity_plane))
    and sanity_x.gp(sanity_plane).grade(3).almost_equal(sanity_x.wedge(sanity_plane)),
    "projection_plus_rejection_recovers_x": (projection + rejection).almost_equal(sanity_x),
    "reflection_keeps_projection_flips_rejection": reflection.almost_equal(projection - rejection),
    "gram_schmidt_off_diagonal_max": float(np.max(np.abs(frame_matrix - np.diag(np.diag(frame_matrix))))),
}
assert all(value is True for key, value in checks.items() if isinstance(value, bool))
assert checks["gram_schmidt_off_diagonal_max"] < 1e-9

sanity_path = save_json(checks, ARTIFACT_TOPIC, "checks", "sanity-checks.json", root=ARTIFACT_ROOT)

notebook = nbformat.read(NOTEBOOK_PATH, as_version=4)
markdown_words = sum(len("".join(cell.get("source", "")).split()) for cell in notebook.cells if cell.cell_type == "markdown")
code_cells = sum(1 for cell in notebook.cells if cell.cell_type == "code")
artifact_root = ARTIFACT_ROOT / ARTIFACT_TOPIC
artifact_files = sorted(str(path.relative_to(artifact_root)) for path in artifact_root.rglob("*") if path.is_file())
quality = {
    "markdown_words": markdown_words,
    "code_cells": code_cells,
    "artifact_count": len(artifact_files),
    "artifacts": artifact_files,
    "meets_markdown_gate_2000": markdown_words >= 2000,
    "meets_code_cell_gate_8": code_cells >= 8,
    "meets_artifact_gate_6": len(artifact_files) >= 6,
}
assert quality["meets_markdown_gate_2000"]
assert quality["meets_code_cell_gate_8"]
assert quality["meets_artifact_gate_6"]
quality_path = save_json(quality, ARTIFACT_TOPIC, "checks", "quality-gate.json", root=ARTIFACT_ROOT)
print(f"wrote {sanity_path}")
print(f"wrote {quality_path}")
quality
